# 03 — Cohort-scale design from a batch of variants

**AlleleForge is a research tool. It is not a medical device and provides no medical advice.**

The same `design()` entry point that handles one variant scales to a cohort: resolve each record,
route it to every eligible chemistry, score candidates with calibrated uncertainty, run
population-aware off-target, and rank — then reduce the per-variant `RankedMenu`s to one summary
table. This notebook runs a small synthetic "VCF" end to end against the weight-free stub models, so
it is fully reproducible in CI; point it at a real VCF + hg38 reference and the loop is unchanged.

In [ ]:
import tempfile
from pathlib import Path

from alleleforge.design.designer import design
from alleleforge.genome.reference import ReferenceGenome
from alleleforge.types.edit import EditIntent
from alleleforge.types.sequence import GenomicInterval, Strand

# One engineered locus: a protospacer with an in-window A (ABE-correctable) and an NGG PAM, so the
# router can offer base editing, prime editing, and (for knock-outs) nuclease across the cohort.
PAD = "T" * 20
PROTO = "TTTAAACGTTTTTTTTTTTT"
contig = PAD + PROTO + "TGG" + PAD

fasta = Path(tempfile.mkdtemp()) / "cohort.fa"
fasta.write_text(f">chr2\n{contig}\n")
reference = ReferenceGenome(fasta, build="hg38")

def ref_base(one_based: int) -> str:
    iv = GenomicInterval(chrom="chr2", start=one_based - 1, end=one_based, strand=Strand.PLUS)
    return str(reference.fetch(iv))

print("locus length:", len(contig), "| edit position chr2:26 ref base:", ref_base(26))

## 1. A synthetic cohort

Each row stands in for a VCF record: a sample id, a 1-based position, an alternate allele, and the
editing intent. In practice these come straight from `cyvcf2`/`pysam` iteration over a `.vcf.gz`.

In [ ]:
cohort = [
    {"sample": "S1", "pos": 26, "alt": "G", "intent": EditIntent.INSTALL},
    {"sample": "S2", "pos": 26, "alt": "C", "intent": EditIntent.INSTALL},
    {"sample": "S3", "pos": 26, "alt": "G", "intent": EditIntent.KNOCK_OUT},
]
for rec in cohort:
    rec["variant"] = f"chr2:{rec['pos']}:{ref_base(rec['pos'])}>{rec['alt']}"
print("\n".join(f"{r['sample']}: {r['variant']}  ({r['intent'].value})" for r in cohort))

## 2. Design every sample, then reduce to a summary

`design()` accepts a raw input string and returns a `RankedMenu` carrying ranked candidates and full
provenance. We collect the headline facts per sample — how many chemistries were offered, the best
candidate's chemistry, its calibrated efficiency, and the worst off-target score across the menu.

In [ ]:
rows = []
for rec in cohort:
    menu = design(rec["variant"], reference=reference, intent=rec["intent"], run_offtarget=True)
    best = menu.best
    eff = best.efficiency.value if best and best.efficiency else float("nan")
    worst_ot = max(
        (c.offtarget.worst_score() for c in menu.candidates if c.offtarget is not None),
        default=0.0,
    )
    rows.append({
        "sample": rec["sample"],
        "intent": rec["intent"].value,
        "n_candidates": len(menu.candidates),
        "chemistries": "|".join(sorted({c.chemistry.value for c in menu.candidates})),
        "best": best.chemistry.value if best else "-",
        "best_eff": round(eff, 2),
        "worst_offtarget": round(worst_ot, 3),
    })

hdr = ["sample", "intent", "n_candidates", "chemistries", "best", "best_eff", "worst_offtarget"]
print("  ".join(f"{h:>14}" if h != "chemistries" else f"{h:<28}" for h in hdr))
for r in rows:
    print("  ".join(
        f"{str(r[h]):<28}" if h == "chemistries" else f"{str(r[h]):>14}" for h in hdr
    ))

## 3. Every result is reproducible

Each menu embeds a provenance block — AlleleForge version, seed, reference build, and the model
checkpoints and datasets touched — so a cohort run can be re-derived from its config and seed. This
is what makes a batch result auditable rather than a one-off.

In [ ]:
menu = design(cohort[0]["variant"], reference=reference, intent=cohort[0]["intent"])
prov = menu.provenance
print("alleleforge version :", prov.alleleforge_version)
print("seed                :", prov.seed)
print("reference build     :", prov.reference_build)
print("models invoked      :", [m.name for m in prov.models])
assert len(rows) == len(cohort)

For a real cohort, stream records with `cyvcf2`, pass a gnomAD database and per-sample phased
haplotypes to `design(..., gnomad=..., haplotypes=...)` for population- and haplotype-aware
off-target, and write each `RankedMenu` plus its provenance sidecar with the Phase 11 report
exporters. The orchestration above does not change — only the data sources do.